In [2]:
import sys
import os

# Add project root to Python path
sys.path.insert(0, os.path.abspath("../.."))

In [3]:
import os, sys
from src.utils.config import config
from pyspark.sql import SparkSession
import time
from src.utils.config import config
import re
import html
from pyspark.sql.functions import udf 
from pyspark.sql.types import StringType
from pyspark.sql.functions import col



os.environ["PYSPARK_PYTHON"] = "python"
os.environ["PYSPARK_DRIVER_PYTHON"] = "python"
os.environ["HADOOP_HOME"] = "C:/hadoop"
os.environ["PATH"] = os.environ["PATH"] + ";C:/hadoop/bin"


In [4]:

project_root = os.path.abspath("../..")
print(project_root)

if project_root not in sys.path:
    sys.path.insert(0, project_root)



c:\Users\Akhil\OneDrive\Documents\reddit-rag-analytics-pipeline


In [5]:
# importing spark session
spark = SparkSession.builder \
    .appName("reddit rag pipeline") \
    .master("local[*]") \
    .getOrCreate()

In [ ]:
subreddit = config.subreddits[1]
file_path = config.posts_path(subreddit)

# Point directly to posts.json
file_path_str = str(file_path / "posts.json").replace("\\", "/")
print(file_path_str)

In [ ]:
df = spark.read.option("multiline", "true").json(file_path_str)
df.show(20)

In [ ]:
df_req_col = df.select(
    "id",
    "title", 
    "selftext",
    "author",
    "score",
    "upvote_ratio",
    "num_comments",
    "created_utc",
    "url",  
    "permalink",
    "subreddit",
    "subreddit_subscribers",
    "over_18",
    "link_flair_text"
).dropDuplicates(["id"])

In [ ]:
df_req_col.show(5)

In [ ]:
def clean_text(text):
    if not text:
        return ""
    
    # Step 2 - decode HTML entities (your turn)
    text = html.unescape(text)

    # bold / italic
    text = re.sub(r"\*\*(.*?)\*\*", r"\1", text)
    text = re.sub(r"\*(.*?)\*", r"\1", text)
    # strikethrough
    text = re.sub(r"~~(.*?)~~", r"\1", text)

    # inline code
    text = re.sub(r"`(.*?)`", r"\1", text)

    # links
    text = re.sub(r"\[(.*?)\]\(.*?\)", r"\1", text)

    # Step 1 - remove URLs
    text = re.sub(r'http\S+', '', text)
    
    # Step 4 - replace \n with space (your turn)
    text = text.replace("\n", " ")
    
    # Remove multiple spaces
    text = re.sub(r' +', ' ', text)

    # Step 5 - strip extra whitespace (your turn)
    text = text.strip()
    

    return text

In [ ]:

clean_text_udf = udf(clean_text, StringType())
df_clean_text = df_req_col.withColumn("clean_selftext", clean_text_udf("selftext"))

# Show first 5 rows after cleaning to verify results
df_clean_text.show(20, truncate=False)


In [ ]:
df_final = df_clean_text.select(
    "id",
    "title", 
    "clean_selftext",
    "author",
    "score",
    "upvote_ratio",
    "num_comments",
    "created_utc",
    "url",  
    "permalink",
    "subreddit",
    "subreddit_subscribers",
    "over_18",
    "link_flair_text"
).dropDuplicates(["id"])

df_final.createOrReplaceTempView("df_final")

In [ ]:
posts_output_path = str(config.processed_path(subreddit) / "posts" ).replace("\\", "/")
print(posts_output_path)

In [ ]:
df_final.write.mode("overwrite").parquet(posts_output_path)

In [ ]:
print(config.processed_path(subreddit))

##COMMENTSS

In [ ]:
subreddit = config.subreddits[1]
comments_file_path = config.comments_path(subreddit)

# Point directly to posts.json
comments_file_path_str = str(comments_file_path / "comments.json").replace("\\", "/")
print(comments_file_path_str)

In [ ]:
df_comments = spark.read.option("multiline", "true").json(comments_file_path_str)

In [ ]:

df_comments_req_cols = df_comments.select(
    "id", 
    "body", 
    "author", 
    "score", 
    "created_utc", 
    "depth", 
    "parent_id", 
    "parent_post_id", 
    "permalink", 
    "subreddit"
).dropDuplicates(["id"])


df_clean_comments  = df_comments_req_cols.filter(col("depth") == 0).withColumn("clean_body", clean_text_udf("body"))


df_comments_final= df_clean_comments.select(
    "id", 
    "clean_body", 
    "author", 
    "score", 
    "created_utc", 
    "depth", 
    "parent_id", 
    "parent_post_id", 
    "permalink", 
    "subreddit"
).dropDuplicates(["id"])


In [ ]:
psots_output_path = str(config.processed_path(subreddit) / "posts" ).replace("\\", "/")
print(psots_output_path)

In [ ]:
comments_output_path = str(config.processed_path(subreddit) / "comments" ).replace("\\", "/")
print(comments_output_path)

In [ ]:
df_comments_final.write.mode("overwrite").parquet(comments_output_path)

In [ ]:
import pandas as pd
from src.embeddings.chunker import chunk_posts, chunk_comments
from src.utils.snowflake_client import get_snowflake_connection


with get_snowflake_connection() as conn:
    df = pd.read_sql("SELECT * FROM REDDIT_RAG.MARTS.FCT_POSTS LIMIT 50", conn)

post_docs = chunk_posts(df)
comment_docs = chunk_comments(df)

print(f"Post chunks: {len(post_docs)}")
print(f"Comment chunks: {len(comment_docs)}")
print(f"\nSample post chunk:")
print(post_docs[0].page_content[:200])
print(f"\nMetadata: {post_docs[0].metadata}")

ModuleNotFoundError: No module named 'langchain.text_splitter'